# Regresión Lineal Múltiple - Guía de Estudio
## Caso: Predicción de Ventas de Vehículos

**Objetivo**: Predecir la variable **Ventas** basada en **Precio** y **Kilometraje**

---

## Librerías Necesarias

```bash
pip install pandas numpy matplotlib seaborn statsmodels scikit-learn scipy
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

---

## BLOQUE 1: LINEALIDAD

### Teoría

La **linealidad** es el supuesto fundamental de la regresión lineal. Exige que la relación entre las variables independientes (X) y la variable dependiente (y) sea **lineal**.

**¿Por qué es importante?**
- Si la relación no es lineal, el modelo no capturará correctamente el patrón
- Los coeficientes estimados serán sesgados
- Las predicciones serán inexactas

**Forma de detectarla**: Usamos diagramas de dispersión (scatterplots) para visualizar la relación entre cada predictor y la variable dependiente.

In [ ]:
# ============================================================
# ESCENARIO 1: FALLA - Relación CUADRÁTICA (No lineal)
# ============================================================

np.random.seed(42)
n = 200

# Variables independientes
precio_cuadratico = np.linspace(10000, 100000, n)
kilometraje_cuadratico = np.linspace(0, 200000, n)

# Variable dependiente: relación CUADRÁTICA con el precio
# La relación es: Ventas = a + b*precio² (no lineal)
ventas_cuadratico = 50 - 0.000003 * (precio_cuadratico**2) + 0.0001 * kilometraje_cuadratico + np.random.normal(0, 5, n)

df_cuadratico = pd.DataFrame({
    'Precio': np.round(precio_cuadratico, 2),
    'Kilometraje': np.round(kilometraje_cuadratico, 2),
    'Ventas': np.round(ventas_cuadratico, 2)
})

# Visualización - Relación NO lineal
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df_cuadratico, x='Precio', y='Ventas', ax=axes[0], alpha=0.6, color='#e74c3c')
axes[0].set_title('ESCENARIO 1 FALLA: Relación CUADRÁTICA\n(Patrón CURVO - No lineal)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Precio ($)')
axes[0].set_ylabel('Ventas (unidades)')

sns.scatterplot(data=df_cuadratico, x='Kilometraje', y='Ventas', ax=axes[1], alpha=0.6, color='#e74c3c')
axes[1].set_title('Relación Kilometraje vs Ventas', fontsize=12)
axes[1].set_xlabel('Kilometraje (km)')
axes[1].set_ylabel('Ventas (unidades)')

plt.tight_layout()
plt.show()

print("⚠️ OBSERVACIÓN: El gráfico muestra un PATRÓN CURVO (U invertida)")
print("   Esto viola el supuesto de linealidad.")
print("   Un modelo lineal NO capturará correctamente esta relación.")

In [ ]:
# Ajustar modelo lineal a datos cuadráticos (para mostrar el error)
X_fail = df_cuadratico[['Precio', 'Kilometraje']]
y_fail = df_cuadratico['Ventas']

X_fail_const = sm.add_constant(X_fail)
modelo_fail = sm.OLS(y_fail, X_fail_const).fit()

print("=" * 60)
print("MODELO LINEAL AJUSTADO A DATOS CUADRÁTICOS (FALLA)")
print("=" * 60)
print(f"R² Ajustado: {modelo_fail.rsquared_adj:.4f}")
print(f"   → El modelo lineal captura muy poco de la relación real")
print(f"\nCoeficientes:")
for var, coef in modelo_fail.params.items():
    print(f"   {var}: {coef:.6f}")

In [ ]:
# ============================================================
# ESCENARIO 2: PASA - Relación LINEAL
# ============================================================

np.random.seed(123)
n = 200

# Variables independientes
precio_lineal = np.linspace(10000, 100000, n)
kilometraje_lineal = np.linspace(0, 200000, n)

# Variable dependiente: relación LINEAL
# Ventas = a - b*precio - c*kilometraje (relación lineal negativa)
ventas_lineal = 100 - 0.0005 * precio_lineal - 0.0002 * kilometraje_lineal + np.random.normal(0, 5, n)

df_lineal = pd.DataFrame({
    'Precio': np.round(precio_lineal, 2),
    'Kilometraje': np.round(kilometraje_lineal, 2),
    'Ventas': np.round(ventas_lineal, 2)
})

# Visualización - Relación lineal
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df_lineal, x='Precio', y='Ventas', ax=axes[0], alpha=0.6, color='#27ae60')
axes[0].set_title('ESCENARIO 2 PASA: Relación LINEAL\n(Patrón RECTO - Cumple supuesto)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Precio ($)')
axes[0].set_ylabel('Ventas (unidades)')

sns.scatterplot(data=df_lineal, x='Kilometraje', y='Ventas', ax=axes[1], alpha=0.6, color='#27ae60')
axes[1].set_title('Relación Kilometraje vs Ventas', fontsize=12)
axes[1].set_xlabel('Kilometraje (km)')
axes[1].set_ylabel('Ventas (unidades)')

plt.tight_layout()
plt.show()

print("✅ OBSERVACIÓN: El gráfico muestra un PATRÓN RECTO (lineal)")
print("   Esto cumple el supuesto de linealidad.")
print("   Un modelo lineal capturará correctamente esta relación.")

In [ ]:
# Ajustar modelo lineal a datos lineales (para mostrar éxito)
X_pass = df_lineal[['Precio', 'Kilometraje']]
y_pass = df_lineal['Ventas']

X_pass_const = sm.add_constant(X_pass)
modelo_pass = sm.OLS(y_pass, X_pass_const).fit()

print("=" * 60)
print("MODELO LINEAL AJUSTADO A DATOS LINEALES (PASA)")
print("=" * 60)
print(f"R² Ajustado: {modelo_pass.rsquared_adj:.4f}")
print(f"   → El modelo lineal captura correctamente la relación")
print(f"\nCoeficientes:")
for var, coef in modelo_pass.params.items():
    print(f"   {var}: {coef:.6f}")

---

## BLOQUE 2: NORMALIDAD DE LOS RESIDUOS

### Teoría

Los **residuos** (errores) son la diferencia entre el valor observado y el valor predicho: $e = y - \hat{y}$

**¿Por qué es importante la normalidad?**
- Intervalos de confianza válidos
- Pruebas de hipótesis confiables
- Estimadores BLUE (Best Linear Unbiased Estimators)

**Métodos de detección**:
1. Histograma de residuos
2. Gráfico Q-Q (Quantile-Quantile)
3. Test de Shapiro-Wilk (prueba formal)

In [ ]:
# ============================================================
# ESCENARIO 1: FALLA - Residuos con SESGO EXTREMO
# ============================================================

np.random.seed(42)
n = 200

# Datos con relación lineal
precio_normal = np.linspace(10000, 100000, n)
kilometraje_normal = np.linspace(0, 200000, n)
ventas_normal = 100 - 0.0005 * precio_normal - 0.0002 * kilometraje_normal + np.random.normal(0, 5, n)

df_normal = pd.DataFrame({
    'Precio': np.round(precio_normal, 2),
    'Kilometraje': np.round(kilometraje_normal, 2),
    'Ventas': np.round(ventas_normal, 2)
})

# Ajustar modelo
X_n = df_normal[['Precio', 'Kilometraje']]
y_n = df_normal['Ventas']
X_n_const = sm.add_constant(X_n)
modelo_n = sm.OLS(y_n, X_n_const).fit()

# Generar residuos con SESGO EXTREMO (usando distribución exponencial sesgada)
# Esto simula errores sistemáticos en una dirección
residuos_fallo = modelo_n.resid + np.random.exponential(10, n)  # Sesgo positivo

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(residuos_fallo, bins=30, edgecolor='black', alpha=0.7, color='#e74c3c')
axes[0].axvline(x=0, color='black', linestyle='--', linewidth=2)
axes[0].set_title('ESCENARIO 1 FALLA: Residuos con SESGO EXTREMO\n(Histograma asimétrico)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Residuos')
axes[0].set_ylabel('Frecuencia')

# Q-Q Plot
sm.qqplot(residuos_fallo, line='45', ax=axes[1], color='#e74c3c')
axes[1].set_title('Gráfico Q-Q\n(Desviación de la línea diagonal)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Test de Shapiro-Wilk
shapiro_stat, shapiro_p = stats.shapiro(residuos_fallo)
print(f"\n📊 Test de Shapiro-Wilk:")
print(f"   Estadístico: {shapiro_stat:.4f}")
print(f"   p-value: {shapiro_p:.6f}")
if shapiro_p < 0.05:
    print(f"   ⚠️ p-value < 0.05 → RECHAZAMOS la normalidad")
else:
    print(f"   ✅ p-value >= 0.05 → No rechazamos la normalidad")

In [ ]:
# ============================================================
# ESCENARIO 2: PASA - Residuos NORMALES
# ============================================================

# Usamos los residuos DEL MODELO (que ya son normales por construcción)
residuos_paso = modelo_n.resid  # Residuos con distribución normal

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(residuos_paso, bins=30, edgecolor='black', alpha=0.7, color='#27ae60')
axes[0].axvline(x=0, color='black', linestyle='--', linewidth=2)
axes[0].set_title('ESCENARIO 2 PASA: Residuos NORMALES\n(Histograma simétrico en forma de campana)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Residuos')
axes[0].set_ylabel('Frecuencia')

# Q-Q Plot
sm.qqplot(residuos_paso, line='45', ax=axes[1], color='#27ae60')
axes[1].set_title('Gráfico Q-Q\n(Puntos sobre la línea diagonal)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Test de Shapiro-Wilk
shapiro_stat2, shapiro_p2 = stats.shapiro(residuos_paso)
print(f"\n📊 Test de Shapiro-Wilk:")
print(f"   Estadístico: {shapiro_stat2:.4f}")
print(f"   p-value: {shapiro_p2:.6f}")
if shapiro_p2 < 0.05:
    print(f"   ⚠️ p-value < 0.05 → RECHAZAMOS la normalidad")
else:
    print(f"   ✅ p-value >= 0.05 → No rechazamos la normalidad")

---

## BLOQUE 3: HOMOCEDASTICIDAD (Varianza Constante)

### Teoría

La **homocedasticidad** significa que la varianza de los residuos es **constante** a lo largo de todos los valores predichos.

**¿Qué es la heterocedasticidad?**
- Cuando la varianza de los errores **cambia** con el valor de X
- Se manifiesta como una "forma de abanico" o "cono" en el gráfico

**Problemas que causa**:
- Estimadores ineficientes
- Intervalos de confianza incorrectos
- Errores estándar subestimados (falsamente precisos)

**Gráfico de detección**: Residuos vs Valores Predichos

In [ ]:
# ============================================================
# ESCENARIO 1: FALLA - HETEROCEDASTICIDAD
# ============================================================

np.random.seed(42)
n = 200

# Variables
precio_hetero = np.linspace(10000, 100000, n)
kilometraje_hetero = np.linspace(0, 200000, n)

# La varianza AUMENTA con el precio (heterocedasticidad)
varianza = 1 + (precio_hetero / 10000)  # Varianza aumenta con el precio
ruido = np.array([np.random.normal(0, v) for v in varianza])

ventas_hetero = 100 - 0.0005 * precio_hetero - 0.0002 * kilometraje_hetero + ruido

df_hetero = pd.DataFrame({
    'Precio': np.round(precio_hetero, 2),
    'Kilometraje': np.round(kilometraje_hetero, 2),
    'Ventas': np.round(ventas_hetero, 2)
})

# Ajustar modelo
X_h = df_hetero[['Precio', 'Kilometraje']]
y_h = df_hetero['Ventas']
X_h_const = sm.add_constant(X_h)
modelo_h = sm.OLS(y_h, X_h_const).fit()

# Residuos
residuos_hetero = modelo_h.resid
predichos_hetero = modelo_h.fittedvalues

# Visualización
plt.figure(figsize=(10, 6))
plt.scatter(predichos_hetero, residuos_hetero, alpha=0.6, color='#e74c3c', edgecolors='white', s=50)
plt.axhline(y=0, color='black', linestyle='--', linewidth=2)
plt.xlabel('Valores Predichos', fontsize=12)
plt.ylabel('Residuos', fontsize=12)
plt.title('ESCENARIO 1 FALLA: HETEROCEDASTICIDAD\n(Forma de ABANICO - Varianza NO constante)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("⚠️ OBSERVACIÓN: Los residuos muestran una 'FORMA DE ABANICO'")
print("   La dispersión AUMENTA a medida que aumentan los valores predichos.")
print("   Esto viola el supuesto de homocedasticidad.")

In [ ]:
# Test de Breusch-Pagan (prueba formal de heterocedasticidad)
from statsmodels.stats.diagnostic import het_breuschpagan

bp_stat, bp_pvalue, _, _ = het_breuschpagan(residuos_hetero, X_h_const)

print("=" * 50)
print("Test de Breusch-Pagan")
print("=" * 50)
print(f"   Estadístico LM: {bp_stat:.4f}")
print(f"   p-value: {bp_pvalue:.6f}")
if bp_pvalue < 0.05:
    print(f"   ⚠️ p-value < 0.05 → RECHAZAMOS homocedasticidad")
else:
    print(f"   ✅ p-value >= 0.05 → No rechazamos homocedasticidad")

In [ ]:
# ============================================================
# ESCENARIO 2: PASA - HOMOCEDASTICIDAD
# ============================================================

np.random.seed(123)
n = 200

# Variables
precio_homo = np.linspace(10000, 100000, n)
kilometraje_homo = np.linspace(0, 200000, n)

# Varianza CONSTANTE
ventas_homo = 100 - 0.0005 * precio_homo - 0.0002 * kilometraje_homo + np.random.normal(0, 5, n)

df_homo = pd.DataFrame({
    'Precio': np.round(precio_homo, 2),
    'Kilometraje': np.round(kilometraje_homo, 2),
    'Ventas': np.round(ventas_homo, 2)
})

# Ajustar modelo
X_ho = df_homo[['Precio', 'Kilometraje']]
y_ho = df_homo['Ventas']
X_ho_const = sm.add_constant(X_ho)
modelo_ho = sm.OLS(y_ho, X_ho_const).fit()

# Residuos
residuos_homo = modelo_ho.resid
predichos_homo = modelo_ho.fittedvalues

# Visualización
plt.figure(figsize=(10, 6))
plt.scatter(predichos_homo, residuos_homo, alpha=0.6, color='#27ae60', edgecolors='white', s=50)
plt.axhline(y=0, color='black', linestyle='--', linewidth=2)
plt.xlabel('Valores Predichos', fontsize=12)
plt.ylabel('Residuos', fontsize=12)
plt.title('ESCENARIO 2 PASA: HOMOCEDASTICIDAD\n(Dispersión CONSTANTE - Varianza estable)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✅ OBSERVACIÓN: Los residuos tienen una dispersión CONSTANTE")
print("   No hay patrón de 'abanico' o 'cono'.")
print("   Esto cumple el supuesto de homocedasticidad.")

In [ ]:
# Test de Breusch-Pagan para homocedasticidad
bp_stat2, bp_pvalue2, _, _ = het_breuschpagan(residuos_homo, X_ho_const)

print("=" * 50)
print("Test de Breusch-Pagan")
print("=" * 50)
print(f"   Estadístico LM: {bp_stat2:.4f}")
print(f"   p-value: {bp_pvalue2:.6f}")
if bp_pvalue2 < 0.05:
    print(f"   ⚠️ p-value < 0.05 → RECHAZAMOS homocedasticidad")
else:
    print(f"   ✅ p-value >= 0.05 → No rechazamos homocedasticidad")

---

## BLOQUE 4: MULTICOLINEALIDAD (VIF)

### Teoría

La **multicolinealidad** ocurre cuando dos o más variables independientes están altamente correlacionadas entre sí.

**¿Por qué es un problema?**
- Dificulta identificar el efecto individual de cada predictor
- Increase the variance of coefficient estimates
- Makes estimates unstable and sensitive to small changes in data

**Solución: Variance Inflation Factor (VIF)**
- Mide cuánto se inflate la varianza de un coeficiente debido a la correlación
- **VIF > 10**: Alta multicolinealidad (problema)
- **VIF < 5**: Aceptable
- **VIF = 1**: Sin correlación

In [ ]:
# ============================================================
# ESCENARIO 1: FALLA - ALTA MULTICOLINEALIDAD (VIF > 10)
# ============================================================

np.random.seed(42)
n = 200

# Crear variables ALTAMENTE correlacionadas
# Kilometraje y Kilometraje_Años están relacionados (misma información)
precio_mult = np.linspace(10000, 100000, n)
kilometraje_mult = np.linspace(0, 200000, n)

# Kilometraje_Años es una transformación de Kilometraje (alta correlación)
kilometraje_anios = kilometraje_mult / 15000 + np.random.normal(0, 0.5, n)

ventas_mult = 100 - 0.0005 * precio_mult - 0.5 * kilometraje_anios + np.random.normal(0, 5, n)

df_mult = pd.DataFrame({
    'Precio': np.round(precio_mult, 2),
    'Kilometraje': np.round(kilometraje_mult, 2),
    'Kilometraje_Años': np.round(kilometraje_anios, 2),
    'Ventas': np.round(ventas_mult, 2)
})

print("=" * 60)
print("ESCENARIO 1 FALLA: Alta Multicolinealidad")
print("=" * 60)

# Calcular correlación entre predictores
print("\n📊 Matriz de correlación entre predictores:")
corr_matrix = df_mult[['Precio', 'Kilometraje', 'Kilometraje_Años']].corr()
print(corr_matrix.round(3))

print(f"\n⚠️ Correlación entre Kilometraje y Kilometraje_Años: {corr_matrix.loc['Kilometraje', 'Kilometraje_Años']:.3f}")
print("   → MUY ALTA (casi 1.0) - Misma información duplicada")

In [ ]:
# Calcular VIF
X_vif_fail = df_mult[['Precio', 'Kilometraje', 'Kilometraje_Años']]

vif_data = pd.DataFrame()
vif_data['Variable'] = X_vif_fail.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif_fail.values, i) for i in range(X_vif_fail.shape[1])]

print("\n📊 Factor de Inflación de Varianza (VIF):")
print(vif_data.to_string(index=False))

print("\n⚠️ INTERPRETACIÓN:")
for _, row in vif_data.iterrows():
    if row['VIF'] > 10:
        print(f"   {row['Variable']}: VIF = {row['VIF']:.2f} → PROBLEMA (VIF > 10)")
    elif row['VIF'] > 5:
        print(f"   {row['Variable']}: VIF = {row['VIF']:.2f} → Precaución (VIF > 5)")
    else:
        print(f"   {row['Variable']}: VIF = {row['VIF']:.2f} → Aceptable")

In [ ]:
# ============================================================
# ESCENARIO 2: PASA - Variables INDEPENDIENTES (VIF bajo)
# ============================================================

np.random.seed(123)
n = 200

# Variables independientes (sin correlación entre ellas)
precio_ind = np.linspace(10000, 100000, n)
kilometraje_ind = np.linspace(0, 200000, n)
anios_uso = np.random.randint(0, 20, n)  # Años de uso (independiente)

ventas_ind = 100 - 0.0005 * precio_ind - 0.3 * kilometraje_ind - 2 * anios_uso + np.random.normal(0, 5, n)

df_ind = pd.DataFrame({
    'Precio': np.round(precio_ind, 2),
    'Kilometraje': np.round(kilometraje_ind, 2),
    'Años_Uso': anios_uso,
    'Ventas': np.round(ventas_ind, 2)
})

print("=" * 60)
print("ESCENARIO 2 PASA: Variables Independientes")
print("=" * 60)

# Calcular correlación
print("\n📊 Matriz de correlación entre predictores:")
corr_matrix2 = df_ind[['Precio', 'Kilometraje', 'Años_Uso']].corr()
print(corr_matrix2.round(3))

In [ ]:
# Calcular VIF
X_vif_pass = df_ind[['Precio', 'Kilometraje', 'Años_Uso']]

vif_data2 = pd.DataFrame()
vif_data2['Variable'] = X_vif_pass.columns
vif_data2['VIF'] = [variance_inflation_factor(X_vif_pass.values, i) for i in range(X_vif_pass.shape[1])]

print("\n📊 Factor de Inflación de Varianza (VIF):")
print(vif_data2.to_string(index=False))

print("\n✅ INTERPRETACIÓN:")
for _, row in vif_data2.iterrows():
    if row['VIF'] > 10:
        print(f"   {row['Variable']}: VIF = {row['VIF']:.2f} → PROBLEMA (VIF > 10)")
    elif row['VIF'] > 5:
        print(f"   {row['Variable']}: VIF = {row['VIF']:.2f} → Precaución (VIF > 5)")
    else:
        print(f"   {row['Variable']}: VIF = {row['VIF']:.2f} → Aceptable")

print("\n   → Todas las variables tienen VIF < 5 → Sin multicolinealidad")

---

## BLOQUE 5: EVALUACIÓN FINAL (R² y MSE)

### Teoría

**R² (R-cuadrado o Coeficiente de Determinación)**
- Mide la proporción de varianza explicada por el modelo
- Rango: 0 a 1 (0% a 100%)
- **Mayor R² = Mejor ajuste**

**MSE (Error Cuadrático Medio)**
- Mide el error promedio al cuadrado
- Penaliza errores grandes más que errores pequeños
- **Menor MSE = Mejor modelo**

**Diferencia**: R² mide *precisión* (bondad de ajuste), MSE mide *error absoluto* (magnitud del error)

In [ ]:
# ============================================================
# CÁLCULO DE R² y MSE (usando el dataset lineal)
# ============================================================

# Usar los datos lineales del Bloque 1
X_eval = df_lineal[['Precio', 'Kilometraje']]
y_eval = df_lineal['Ventas']

# Ajustar modelo
modelo_eval = LinearRegression()
modelo_eval.fit(X_eval, y_eval)
y_pred_eval = modelo_eval.predict(X_eval)

print("=" * 60)
print("EVALUACIÓN DEL MODELO")
print("=" * 60)

# ============ CÁLCULO MANUAL ============

# R² Manual
ss_res = np.sum((y_eval - y_pred_eval) ** 2)  # Suma de cuadrados residual
ss_tot = np.sum((y_eval - np.mean(y_eval)) ** 2)  # Suma de cuadrados total
r2_manual = 1 - (ss_res / ss_tot)

# MSE Manual
n_samples = len(y_eval)
mse_manual = np.sum((y_eval - y_pred_eval) ** 2) / n_samples

print("\n📊 CÁLCULO MANUAL:")
print(f"   R² (manual): {r2_manual:.4f}")
print(f"   MSE (manual): {mse_manual:.4f}")

# ============ CÁLCULO CON SKLEARN ============

r2_sklearn = r2_score(y_eval, y_pred_eval)
mse_sklearn = mean_squared_error(y_eval, y_pred_eval)
rmse_sklearn = np.sqrt(mse_sklearn)

print("\n📊 CÁLCULO CON SKLEARN:")
print(f"   R² (sklearn): {r2_sklearn:.4f}")
print(f"   MSE (sklearn): {mse_sklearn:.4f}")
print(f"   RMSE (sklearn): {rmse_sklearn:.4f}")

print("\n📊 CÁLCULO CON STATSMODELS:")
X_eval_const = sm.add_constant(X_eval)
modelo_sm = sm.OLS(y_eval, X_eval_const).fit()
print(f"   R² (statsmodels): {modelo_sm.rsquared:.4f}")
print(f"   R² Ajustado: {modelo_sm.rsquared_adj:.4f}")

In [ ]:
# Comparación visual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Valores reales vs predichos
axes[0].scatter(y_eval, y_pred_eval, alpha=0.6, color='#3498db', edgecolors='white')
axes[0].plot([y_eval.min(), y_eval.max()], [y_eval.min(), y_eval.max()], 'r--', linewidth=2, label='Perfecto')
axes[0].set_xlabel('Valores Reales', fontsize=12)
axes[0].set_ylabel('Valores Predichos', fontsize=12)
axes[0].set_title(f'Valores Reales vs Predichos\nR² = {r2_sklearn:.4f}', fontsize=12, fontweight='bold')
axes[0].legend()

# Residuos
residuos_eval = y_eval - y_pred_eval
axes[1].hist(residuos_eval, bins=30, edgecolor='black', alpha=0.7, color='#9b59b6')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residuos', fontsize=12)
axes[1].set_ylabel('Frecuencia', fontsize=12)
axes[1].set_title(f'Distribución de Residuos\nMSE = {mse_sklearn:.4f}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

---

## CONCLUSIÓN: Impacto en Decisiones de Negocio

### Resumen de Supuestos y su Impacto

| Supuesto | Problema si Falla | Impacto en Negocio |
|----------|-------------------|-------------------|
| **Linealidad** | Predicciones inexactas | Decisiones basadas en tendencias falsas |
| **Normalidad** | Intervalos de confianza inválidos | No confiable al estimar rangos de ventas |
| **Homocedasticidad** | Errores estándar incorrectos | Sobre o subestimar la incertidumbre |
| **Multicolinealidad** | Coeficientes inestables | Dificultad para identificar drivers reales |

### Aplicación al Caso de Ventas de Vehículos

1. **Si el modelo no es lineal**: No confíes en predicciones para precios extremos
2. **Si los residuos no son normales**: Los intervalos de confianza de ventas no son confiables
3. **Si hay heterocedasticidad**: Las predicciones para vehículos de lujo (alta precio) son menos precisas
4. **Si hay multicolinealidad**: No puedes determinar si es el precio o el kilometraje lo que afecta más las ventas

**Recomendación**: Antes de usar un modelo de regresión para decisiones de negocio, SIEMPRE verifica estos supuestos. Un modelo que falla en uno o más supuestos puede llevar a decisiones costosas.